In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)
import sys
sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
import mdata_utils 
import TCR_embedings
from sklearn.preprocessing import StandardScaler
from scipy.sparse import issparse

In [2]:
df = pd.read_csv(r".\Tumor_s_vs_multi\separated df\Tumor\tcr_emb_all.csv")

In [3]:
df

,Unnamed: 0,subtype,clone_status,clone_id_size,MANAscore,X_VDJ_1_cdr3_aa_atchley_0,X_VDJ_1_cdr3_aa_atchley_1,X_VDJ_1_cdr3_aa_atchley_2,X_VDJ_1_cdr3_aa_atchley_3,X_VDJ_1_cdr3_aa_atchley_4,...,X_VJ_1_cdr3_aa_composition_12,X_VJ_1_cdr3_aa_composition_13,X_VJ_1_cdr3_aa_composition_14,X_VJ_1_cdr3_aa_composition_15,X_VJ_1_cdr3_aa_composition_16,X_VJ_1_cdr3_aa_composition_17,X_VJ_1_cdr3_aa_composition_18,X_VJ_1_cdr3_aa_composition_19,VDJ_1_cdr3_aa_length,VJ_1_cdr3_aa_length
0,LT1_AAACGGGCATCGGTTA-1,CD8_Tem,Tumor_multiSite_clone,64,0.512226,0.235605,-0.279393,0.271942,0.278817,0.264461,...,1.195913,0.887956,1.536931,1.390758,-0.245931,-0.940186,-0.225208,-0.827658,0.484995,1.877972
1,LT1_AAAGTAGGTAGCACGA-1,CD8_Trm_exh_L,Tumor_multiSite_clone,2,0.096267,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,-0.662635,0.477979,0.628000,-0.172986,-0.940186,-0.225208,-0.827658,-0.520839,1.167852
2,LT1_AAAGTAGGTCGAAAGC-1,CD8_Tem,Tumor_singleSite_clone,11,0.512226,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,0.791044,-0.743889,-0.444630,-1.267162,1.138391,-0.225208,0.359500,1.993746,2.588093
3,LT1_AACCGCGCAGTGAGTG-1,CD8_Tem,Tumor_singleSite_clone,3,0.512226,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,1.126508,-0.743889,-0.252107,-1.267162,0.338939,4.162457,-0.827658,-1.023756,0.457731
4,LT1_AACTCAGAGGCATTGG-1,CD8_Tem,Tumor_multiSite_clone,79,0.512226,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,1.275603,-0.743889,-1.278897,-1.267162,0.445532,-0.225208,-0.827658,-1.023756,-0.252390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6747,RT3_TGTGTTTTCAGTCCCT-1,CD8_Trm_exh_L,Tumor_multiSite_clone,19,0.512226,0.235605,-0.279393,0.271942,0.278817,0.264461,...,1.618781,1.275603,-0.743889,-0.166541,-1.267162,-0.940186,-0.225208,-0.827658,-0.017922,-0.252390
6748,RT3_TTAGGCACACATTTCT-1,CD8_Trm_exh_H,Tumor_single,1,0.771905,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,-0.662635,1.699847,-0.325449,0.921190,0.247573,-0.225208,-0.827658,0.987912,1.167852
6749,RT3_TTAGGCAGTACTCTCC-1,CD8_Trm_exh_L,Tumor_multiSite_clone,2,0.184881,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,-0.662635,0.811216,-0.065417,-1.267162,-0.940186,4.960215,-0.827658,-0.017922,-0.962511
6750,RT3_TTCGAAGGTGCACGAA-1,CD8_Tem,Tumor_multiSite_clone,20,0.177036,0.235605,-0.279393,0.271942,0.278817,0.264461,...,-0.495558,-0.662635,-0.743889,-0.166541,0.009377,-0.940186,-0.225208,-0.827658,-0.017922,-0.252390


In [4]:
sig_LF_index = [10,20,22,36,38,39,41]

In [5]:
subtype = ["CD8_Teff"]
clone_status = ["Tumor_single"]


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

# mask = df['subtype'].isin(subtype) & df['clone_status'].isin(clone_status)
# df_sub = df[mask].copy().reset_index(drop=True)

df_sub = df

feature_cols = df.columns[5:]
X = df_sub[feature_cols].values
y = df_sub['MANAscore'].values

print(f"Subset: {len(df_sub)} cells  |  Features: {len(feature_cols)}")
print(f"MANAscore range: [{y.min():.4f}, {y.max():.4f}]  mean={y.mean():.4f}")

Subset: 6752 cells  |  Features: 242
MANAscore range: [0.0535, 0.9612]  mean=0.4525


In [7]:
## Linear regression with ALL features (col 5+) ##
scaler_all = StandardScaler()
X_scaled = scaler_all.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

lr_all = LinearRegression()
lr_all.fit(X_train, y_train)

y_pred = lr_all.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
corr, pval = pearsonr(y_test, y_pred)

print("=" * 60)
print(f"Linear Regression — ALL features ({len(feature_cols)})")
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"R²: {r2:.4f}   MSE: {mse:.6f}   Pearson r: {corr:.4f} (p={pval:.4g})")
print("=" * 60)

Linear Regression — ALL features (242)
Train: 5401  |  Test: 1351
R²: -112.9129   MSE: 4.809753   Pearson r: 0.0033 (p=0.9049)


In [8]:
import os

slide_dir = r"C:\Users\a4945\Desktop\TCR_MANA_all_0.1_0.4_out"

slide_features = []
for idx in sig_LF_index:
    fpath = os.path.join(slide_dir, f"feature_list_Z{idx}.txt")
    fl = pd.read_csv(fpath, sep="\t")
    names = fl['names'].dropna().tolist()
    names = [n for n in names if n != "NA"]
    slide_features.extend(names)
    print(f"Z{idx}: {len(names)} features")

slide_features = sorted(set(slide_features))
valid_slide = [f for f in slide_features if f in feature_cols]
missing = [f for f in slide_features if f not in feature_cols]

print(f"\nTotal unique SLIDE features: {len(slide_features)}")
print(f"Matched in df: {len(valid_slide)}")
if missing:
    print(f"Missing from df: {missing}")

Z10: 15 features
Z20: 18 features
Z22: 17 features
Z36: 19 features
Z38: 18 features
Z39: 19 features
Z41: 20 features

Total unique SLIDE features: 46
Matched in df: 45
Missing from df: ['clone_id_size']


In [9]:
## Linear regression with SLIDE-selected features ##
X_slide = df_sub[valid_slide].values

scaler_slide = StandardScaler()
X_slide_scaled = scaler_slide.fit_transform(X_slide)

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_slide_scaled, y, test_size=0.2, random_state=42
)

lr_slide = LinearRegression()
lr_slide.fit(X_tr_s, y_tr_s)

y_pred_s = lr_slide.predict(X_te_s)

r2_s = r2_score(y_te_s, y_pred_s)
mse_s = mean_squared_error(y_te_s, y_pred_s)
corr_s, pval_s = pearsonr(y_te_s, y_pred_s)

print("=" * 60)
print(f"Linear Regression — SLIDE features ({len(valid_slide)})")
print(f"Train: {len(X_tr_s)}  |  Test: {len(X_te_s)}")
print(f"R²: {r2_s:.4f}   MSE: {mse_s:.6f}   Pearson r: {corr_s:.4f} (p={pval_s:.4g})")
print("=" * 60)

print(f"\n--- Comparison ---")
print(f"ALL features  ({len(feature_cols):>3d}):  R²={r2:.4f}  MSE={mse:.6f}  r={corr:.4f}")
print(f"SLIDE features ({len(valid_slide):>3d}):  R²={r2_s:.4f}  MSE={mse_s:.6f}  r={corr_s:.4f}")

Linear Regression — SLIDE features (45)
Train: 5401  |  Test: 1351
R²: 0.0659   MSE: 0.039440   Pearson r: 0.2569 (p=8.338e-22)

--- Comparison ---
ALL features  (242):  R²=-112.9129  MSE=4.809753  r=0.0033
SLIDE features ( 45):  R²=0.0659  MSE=0.039440  r=0.2569


## Classification (binarized MANAscore)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler

threshold = np.median(y)
y_cls = (y >= threshold).astype(int)

print(f"MANAscore threshold (median): {threshold:.4f}")
print(f"Class distribution: 0(low)={np.sum(y_cls==0)}, 1(high)={np.sum(y_cls==1)}")

MANAscore threshold (median): 0.5122
Class distribution: 0(low)=2470, 1(high)=4282


In [11]:
## Logistic regression with ALL features (col 5+) ##
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    scaler_all.fit_transform(X), y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

ros_c = RandomOverSampler(random_state=42)
X_train_c_bal, y_train_c_bal = ros_c.fit_resample(X_train_c, y_train_c)

clf_all = LogisticRegression(max_iter=1000, random_state=42)
clf_all.fit(X_train_c_bal, y_train_c_bal)

y_pred_c = clf_all.predict(X_test_c)
y_prob_c = clf_all.predict_proba(X_test_c)[:, 1]

acc_c = accuracy_score(y_test_c, y_pred_c)
try:
    auc_c = roc_auc_score(y_test_c, y_prob_c)
except ValueError:
    auc_c = np.nan

print("=" * 60)
print(f"Logistic Regression — ALL features ({len(feature_cols)})")
print(f"Train(balanced): {len(X_train_c_bal)}  |  Test: {len(X_test_c)}")
print(f"Accuracy: {acc_c:.4f}   ROC AUC: {auc_c:.4f}")
print("=" * 60)
print(classification_report(y_test_c, y_pred_c,
                            target_names=["MANA_low", "MANA_high"],
                            zero_division=0))

Logistic Regression — ALL features (242)
Train(balanced): 6850  |  Test: 1351
Accuracy: 0.5959   ROC AUC: 0.6140
              precision    recall  f1-score   support

    MANA_low       0.46      0.63      0.53       494
   MANA_high       0.73      0.58      0.64       857

    accuracy                           0.60      1351
   macro avg       0.59      0.60      0.59      1351
weighted avg       0.63      0.60      0.60      1351



In [12]:
## Logistic regression with SLIDE-selected features ##
X_tr_sc, X_te_sc, y_tr_sc, y_te_sc = train_test_split(
    scaler_slide.fit_transform(df_sub[valid_slide].values), y_cls,
    test_size=0.2, random_state=42, stratify=y_cls
)

ros_sc = RandomOverSampler(random_state=42)
X_tr_sc_bal, y_tr_sc_bal = ros_sc.fit_resample(X_tr_sc, y_tr_sc)

clf_slide = LogisticRegression(max_iter=1000, random_state=42)
clf_slide.fit(X_tr_sc_bal, y_tr_sc_bal)

y_pred_sc = clf_slide.predict(X_te_sc)
y_prob_sc = clf_slide.predict_proba(X_te_sc)[:, 1]

acc_sc = accuracy_score(y_te_sc, y_pred_sc)
try:
    auc_sc = roc_auc_score(y_te_sc, y_prob_sc)
except ValueError:
    auc_sc = np.nan

print("=" * 60)
print(f"Logistic Regression — SLIDE features ({len(valid_slide)})")
print(f"Train(balanced): {len(X_tr_sc_bal)}  |  Test: {len(X_te_sc)}")
print(f"Accuracy: {acc_sc:.4f}   ROC AUC: {auc_sc:.4f}")
print("=" * 60)
print(classification_report(y_te_sc, y_pred_sc,
                            target_names=["MANA_low", "MANA_high"],
                            zero_division=0))

print(f"\n--- Classification Comparison ---")
print(f"ALL features  ({len(feature_cols):>3d}):  Acc={acc_c:.4f}  AUC={auc_c:.4f}")
print(f"SLIDE features ({len(valid_slide):>3d}):  Acc={acc_sc:.4f}  AUC={auc_sc:.4f}")

Logistic Regression — SLIDE features (45)
Train(balanced): 6850  |  Test: 1351
Accuracy: 0.5211   ROC AUC: 0.5474
              precision    recall  f1-score   support

    MANA_low       0.39      0.57      0.47       494
   MANA_high       0.67      0.49      0.56       857

    accuracy                           0.52      1351
   macro avg       0.53      0.53      0.52      1351
weighted avg       0.57      0.52      0.53      1351


--- Classification Comparison ---
ALL features  (242):  Acc=0.5959  AUC=0.6140
SLIDE features ( 45):  Acc=0.5211  AUC=0.5474
